# 09 — Cascaded student-risk policy (Model B only)

## Policy design
This notebook evaluates a **single active route at a given decision time**, not an OR across every model ever run. A prediction from one stage is not automatically carried forward as a permanent positive label.

| Student age since signup | Route |
|---|---|
| `< 7 days` | Default / defer scoring (no alert) |
| `7–<14 days` | Frozen 7-day snapshot model/features |
| `14–<30 days` | If more than 3 attempts are available by the decision time, use the **latest causal temporal landmark**; otherwise use the frozen 14-day snapshot model |
| `>=30 days` | Frozen 30-day snapshot model/features |

## Important fairness/causality decisions
- **Temporal score:** use the latest eligible landmark available at the decision cutoff, not the maximum score over all landmarks. The maximum/any-landmark rule uses more opportunities to trigger a false positive and is not comparable to a single current-time decision.
- **Thresholds:** tune each route on validation data at its own unit of action (one student), but also evaluate the *combined cascade policy* on validation before freezing it. Do not tune on test.
- **Do not OR independent model predictions by default.** For a fixed decision time, route the student to exactly one model. If the product wants a cumulative “ever alerted” policy, it needs a separate jointly tuned policy and a validation constraint on cumulative student-level precision.
- **Snapshot feature windows are frozen:** the 7-day model always receives day-7 features, not day-10 features; similarly for 14/30. Passing later features to a model trained on an earlier snapshot would be a distribution mismatch.

The current outcome has no event timestamp. All metrics remain against the supplied label, not verified future dropout.

In [1]:
from pathlib import Path
import json, warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, average_precision_score, roc_auc_score, confusion_matrix
from IPython.display import display
ROOT_CANDIDATES=[Path('.'),Path('..'),Path('/mnt/data/student_success_assistant')]
ROOT=next((p.resolve() for p in ROOT_CANDIDATES if (p/'data'/'students.csv').exists()),Path.cwd())
RICH=ROOT/'outputs'/'rich_snapshot'; CURATED=ROOT/'data'/'curated'; OUT=ROOT/'outputs'/'cascade'
for p in [OUT/'metrics',OUT/'predictions']: p.mkdir(parents=True,exist_ok=True)
FLOORS=[0.40,0.50,0.60]
print('Project root:',ROOT)

Project root: /Users/abishek/Documents/Projects/student_success_assistant


## 1. Load the shared student split and rich snapshot predictions

In [2]:
with open(RICH/'split_ids.json') as f: split_ids=json.load(f)
split_lookup={sid:sp for sp,ids in split_ids.items() for sid in ids}
preds=pd.read_csv(RICH/'predictions'/'snapshot_validation_test_probabilities.csv')
print('Saved snapshot prediction rows:',len(preds))
display(preds.head())
assert preds.groupby(['horizon_days','model','split']).student_id.nunique().min()>0

Saved snapshot prediction rows: 1800


,student_id,split,horizon_days,model,target_dropout,probability,snapshot_cutoff
0,S001,validation,7,logistic,0,0.160397,2026-07-25
1,S002,validation,7,logistic,0,0.304238,2026-07-18
2,S005,validation,7,logistic,0,0.154938,2026-07-20
3,S009,validation,7,logistic,0,0.272673,2026-07-16
4,S014,validation,7,logistic,1,0.907782,2026-07-22


## 2. Build a causal temporal landmark table

Use the cleaned per-attempt sequence data. At each attempt, the features include the current attempt and history up to that attempt only. The model's student-level target is the supplied eventual label. We evaluate the temporal branch using one latest landmark per student available by day 30, conditional on having **more than 3 attempts by day 30**. This is a fixed, reproducible decision point for the 14–30-day branch. It is not the any-landmark/max-probability policy.

In [3]:
students=pd.read_csv(ROOT/'data'/'students.csv')
students['signup_dt']=pd.to_datetime(students.signup_date,errors='coerce')
outcomes=pd.read_csv(ROOT/'data'/'outcome.csv')
outcomes['target_dropout']=outcomes.dropped_out.astype(str).str.strip().str.lower().map({'yes':1,'no':0})
seq=pd.read_csv(CURATED/'temporal_attempt_sequences.csv')
seq['attempt_dt']=pd.to_datetime(seq.attempt_dt,errors='coerce'); seq['signup_dt']=pd.to_datetime(seq.signup_dt,errors='coerce')
seq=seq.merge(students[['student_id','age_group','country','device']],on='student_id',how='left',validate='many_to_one')
seq=seq.merge(outcomes[['student_id','target_dropout']],on='student_id',how='left',validate='many_to_one',suffixes=('','_outcome'))
seq=seq[seq.attempt_dt.notna() & seq.signup_dt.notna() & (seq.attempt_dt>=seq.signup_dt)].copy()
seq['days_from_signup']=(seq.attempt_dt-seq.signup_dt).dt.total_seconds()/86400
# Use only features known at/after the current attempt. Exclude identifiers, target, semantics, and any future-derived columns.
TEMP_FEATURES=['days_from_signup','gap_since_previous_attempt_days','sequence_index','score_num','valid_score','score_valid','score_invalid_flag','duration_num','valid_duration_min','duration_valid','previous_valid_score','prior_valid_score_count','prior_score_mean_3','prior_score_std_3','prior_score_mean_all','prior_lesson_count','duration_missing_flag','duration_negative_flag','age_group','country','device']
seq=seq.sort_values(['student_id','attempt_dt','attempt_id'])
# Decision time: day 30 after signup. A row is eligible only if the attempt occurred on/before day 30.
seq30=seq[seq.days_from_signup<=30].copy()
count30=seq30.groupby('student_id').size().rename('attempts_by_day30')
latest=seq30.sort_values(['student_id','attempt_dt','attempt_id']).groupby('student_id',as_index=False).tail(1).copy()
latest=latest.merge(count30,on='student_id',how='left',validate='one_to_one')
latest=latest[latest.attempts_by_day30>3].copy()
latest['split']=latest.student_id.map(split_lookup)
print('Temporal branch eligible students (>3 attempts by day 30):',len(latest))
print('Split counts:',latest.split.value_counts().to_dict())
assert latest.student_id.is_unique


Temporal branch eligible students (>3 attempts by day 30): 191
Split counts: {'train': 103, 'test': 50, 'validation': 38}


## 3. Train a small temporal-branch classifier and choose student-level thresholds

For the cascade branch, each student contributes exactly one latest landmark by day 30. This aligns the model's training/evaluation unit with the one-student decision. Do not train on every landmark and then evaluate the maximum over all landmarks unless that separate policy is explicitly intended.

In [4]:
def make_temp_model():
    cats=[c for c in TEMP_FEATURES if latest[c].dtype=='object' or str(latest[c].dtype).startswith(('string','category','bool'))]
    nums=[c for c in TEMP_FEATURES if c not in cats]
    prep=ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median',add_indicator=True)),('scale',StandardScaler())]),nums),('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('oh',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]),cats)],sparse_threshold=0)
    clf=HistGradientBoostingClassifier(max_iter=100,learning_rate=0.05,max_leaf_nodes=5,l2_regularization=3.0,random_state=42)
    return Pipeline([('prep',prep),('model',clf)])

def choose_threshold(y,p,floor):
    best=None
    for t in np.unique(np.r_[0.0,p,1.0]):
        yh=(p>=t).astype(int); pr=precision_score(y,yh,zero_division=0); re=recall_score(y,yh,zero_division=0)
        if pr>=floor and (best is None or (re,pr,t)>(best[0],best[1],best[2])): best=(re,pr,float(t))
    return (best[2],True,best[0],best[1]) if best else (np.nan,False,np.nan,np.nan)

def score_metrics(y,p,t):
    yh=(p>=t).astype(int); tn,fp,fn,tp=confusion_matrix(y,yh,labels=[0,1]).ravel()
    return {'precision':precision_score(y,yh,zero_division=0),'recall':recall_score(y,yh,zero_division=0),'TP':int(tp),'FP':int(fp),'FN':int(fn),'TN':int(tn),'flagged':int(yh.sum()),'flagged_rate':float(yh.mean())}

tr=latest[latest.split.eq('train')]; va=latest[latest.split.eq('validation')]; te=latest[latest.split.eq('test')]
# If a chronological split contains only one class, stop rather than fabricate metrics.
if len(tr) and tr.target_dropout.nunique()==2 and len(va) and va.target_dropout.nunique()==2:
    temp_model=make_temp_model(); temp_model.fit(tr[TEMP_FEATURES],tr.target_dropout.astype(int))
    pv=temp_model.predict_proba(va[TEMP_FEATURES])[:,1]; pt=temp_model.predict_proba(te[TEMP_FEATURES])[:,1]
    temp_val=va[['student_id','target_dropout']].copy(); temp_val['probability']=pv; temp_val['split']='validation'
    temp_test=te[['student_id','target_dropout']].copy(); temp_test['probability']=pt; temp_test['split']='test'
    temp_preds=pd.concat([temp_val,temp_test],ignore_index=True)
    temp_preds.to_csv(OUT/'predictions'/'temporal_latest_landmark_predictions.csv',index=False)
else:
    print('Temporal branch cannot be fit/evaluated on this split: training or validation has only one class. Review split design / data size.')
    temp_model=None; temp_preds=pd.DataFrame(columns=['student_id','target_dropout','probability','split'])


## 4. Route-specific evaluation

Evaluate each stage on its own eligible student cohort. This is the cleanest first audit of the cascade: each row gets one model route, not a union of positive predictions from all models. The `<7 day` default is reported as no alert and therefore has zero recall by design. The 14–30 branch is split into temporal (>3 attempts by day 30) and frozen day-14 snapshot (otherwise). The >30 branch uses frozen day-30 features.

**Important:** this route-wise report is not the same as an “ever flagged by any stage” system-wide metric. If the product later wants cumulative alerts, thresholds must be jointly selected on validation for the full cumulative policy.

In [5]:
# Choose one model for snapshot routes based on validation recall under the precision floor.
# Model choice and threshold selection are performed using validation only.
snapshot_choice=[]
for h in [7,14,30]:
  for floor in FLOORS:
    cand=[]
    for model_name in preds.model.unique():
      v=preds[(preds.horizon_days==h)&(preds.model==model_name)&(preds.split=='validation')]
      if len(v)==0 or v.target_dropout.nunique()<2: continue
      t,ok,vr,vp=choose_threshold(v.target_dropout.to_numpy(),v.probability.to_numpy(),floor)
      if ok: cand.append((vr,vp,model_name,t))
    if cand:
      vr,vp,mname,t=max(cand,key=lambda z:(z[0],z[1]))
      snapshot_choice.append({'route':f'{h}d_snapshot','horizon_days':h,'precision_floor':floor,'model':mname,'threshold':t,'validation_recall':vr,'validation_precision':vp,'validation_feasible':True})
    else:
      snapshot_choice.append({'route':f'{h}d_snapshot','horizon_days':h,'precision_floor':floor,'model':None,'threshold':np.nan,'validation_recall':np.nan,'validation_precision':np.nan,'validation_feasible':False})
choice_df=pd.DataFrame(snapshot_choice)
# Temporal route thresholds use one row per student (latest landmark by day 30).
if not temp_preds.empty:
  for floor in FLOORS:
    v=temp_preds[temp_preds.split=='validation']
    t,ok,vr,vp=choose_threshold(v.target_dropout.to_numpy(),v.probability.to_numpy(),floor) if len(v) and v.target_dropout.nunique()==2 else (np.nan,False,np.nan,np.nan)
    choice_df=pd.concat([choice_df,pd.DataFrame([{'route':'temporal_latest_landmark','horizon_days':30,'precision_floor':floor,'model':'hist_gradient_boosting','threshold':t,'validation_recall':vr,'validation_precision':vp,'validation_feasible':ok}])],ignore_index=True)

display(choice_df); choice_df.to_csv(OUT/'metrics'/'route_threshold_selection.csv',index=False)


,route,horizon_days,precision_floor,model,threshold,validation_recall,validation_precision,validation_feasible
0,7d_snapshot,7,0.4,random_forest,0.296964,0.851852,0.410714,True
1,7d_snapshot,7,0.5,random_forest,0.485869,0.629630,0.531250,True
2,7d_snapshot,7,0.6,logistic,0.907782,0.333333,0.600000,True
3,14d_snapshot,14,0.4,random_forest,0.297834,0.925926,0.431034,True
4,14d_snapshot,14,0.5,hist_gradient_boosting,0.274787,0.814815,0.550000,True
5,14d_snapshot,14,0.6,hist_gradient_boosting,0.406958,0.703704,0.612903,True
6,30d_snapshot,30,0.4,random_forest,0.213617,1.000000,0.428571,True
7,30d_snapshot,30,0.5,logistic,0.416248,0.888889,0.500000,True
8,30d_snapshot,30,0.6,random_forest,0.446399,0.592593,0.640000,True
9,temporal_latest_landmark,30,0.4,hist_gradient_boosting,0.172033,0.750000,0.500000,True


## 5. Test each route at its frozen threshold

The model family and threshold are selected on validation only. Test metrics are reported as a diagnostic of generalisation. Test precision can fall below the floor; that means the route does not meet the operational requirement yet.

In [6]:
route_test=[]
for r in choice_df.to_dict('records'):
  floor=r['precision_floor']; route=r['route']; t=r['threshold']
  if not r['validation_feasible'] or pd.isna(t):
    route_test.append({**r,'test_precision':np.nan,'test_recall':np.nan,'test_TP':np.nan,'test_FP':np.nan,'test_FN':np.nan,'test_TN':np.nan,'test_flagged':np.nan,'test_meets_precision_floor':False,'status':'no feasible validation threshold'})
    continue
  if route=='temporal_latest_landmark':
    test=temp_preds[temp_preds.split=='test']
    y=test.target_dropout.to_numpy(); p=test.probability.to_numpy()
  else:
    h=int(r['horizon_days']); test=preds[(preds.horizon_days==h)&(preds.model==r['model'])&(preds.split=='test')]
    y=test.target_dropout.to_numpy(); p=test.probability.to_numpy()
  if len(y)==0:
    route_test.append({**r,'test_precision':np.nan,'test_recall':np.nan,'test_TP':np.nan,'test_FP':np.nan,'test_FN':np.nan,'test_TN':np.nan,'test_flagged':0,'test_meets_precision_floor':False,'status':'no test predictions'}); continue
  met=score_metrics(y,p,t)
  route_test.append({**r,**{f'test_{k}':v for k,v in met.items()},'test_meets_precision_floor':met['precision']>=floor,'status':'meets floor on test' if met['precision']>=floor else 'fails floor on test','test_n_students':len(y),'test_positive_count':int(y.sum())})
route_df=pd.DataFrame(route_test); display(route_df)
route_df.to_csv(OUT/'metrics'/'route_test_metrics.csv',index=False)


,route,horizon_days,precision_floor,model,threshold,validation_recall,validation_precision,validation_feasible,test_precision,test_recall,test_TP,test_FP,test_FN,test_TN,test_flagged,test_flagged_rate,test_meets_precision_floor,status,test_n_students,test_positive_count
0,7d_snapshot,7,0.4,random_forest,0.296964,0.851852,0.410714,True,0.366667,0.709677,22.0,38.0,9.0,31.0,60.0,0.60,False,fails floor on test,100.0,31.0
1,7d_snapshot,7,0.5,random_forest,0.485869,0.629630,0.531250,True,0.394737,0.483871,15.0,23.0,16.0,46.0,38.0,0.38,False,fails floor on test,100.0,31.0
2,7d_snapshot,7,0.6,logistic,0.907782,0.333333,0.600000,True,0.625000,0.322581,10.0,6.0,21.0,63.0,16.0,0.16,True,meets floor on test,100.0,31.0
3,14d_snapshot,14,0.4,random_forest,0.297834,0.925926,0.431034,True,0.375000,0.677419,21.0,35.0,10.0,34.0,56.0,0.56,False,fails floor on test,100.0,31.0
4,14d_snapshot,14,0.5,hist_gradient_boosting,0.274787,0.814815,0.550000,True,0.529412,0.580645,18.0,16.0,13.0,53.0,34.0,0.34,True,meets floor on test,100.0,31.0
5,14d_snapshot,14,0.6,hist_gradient_boosting,0.406958,0.703704,0.612903,True,0.551724,0.516129,16.0,13.0,15.0,56.0,29.0,0.29,False,fails floor on test,100.0,31.0
6,30d_snapshot,30,0.4,random_forest,0.213617,1.000000,0.428571,True,0.424242,0.903226,28.0,38.0,3.0,31.0,66.0,0.66,True,meets floor on test,100.0,31.0
7,30d_snapshot,30,0.5,logistic,0.416248,0.888889,0.500000,True,0.481481,0.838710,26.0,28.0,5.0,41.0,54.0,0.54,False,fails floor on test,100.0,31.0
8,30d_snapshot,30,0.6,random_forest,0.446399,0.592593,0.640000,True,0.518519,0.451613,14.0,13.0,17.0,56.0,27.0,0.27,False,fails floor on test,100.0,31.0
9,temporal_latest_landmark,30,0.4,hist_gradient_boosting,0.172033,0.750000,0.500000,True,0.238095,0.714286,5.0,16.0,2.0,27.0,21.0,0.42,False,fails floor on test,50.0,7.0


## 6. Cascade policy: safe operating rules

This notebook intentionally does **not** label a student positive merely because any independent model ever predicted positive. The runtime rule should be:

```text
if age_days < 7:
    defer / default
elif age_days < 14:
    score with frozen day-7 snapshot model and its calibrated policy
elif age_days < 30:
    if attempts_observed_by_now > 3:
        score using latest causal temporal landmark available now
    else:
        score using frozen day-14 snapshot model and its policy
else:
    score using frozen day-30 snapshot model and its policy
```

At a given decision time, exactly one route is active. A later decision may update the earlier one, but it should replace the prior risk state rather than automatically OR-ing every historical model output. If the business wants “ever flagged”, evaluate that as a distinct cumulative policy and jointly tune the stage thresholds on validation so the **cumulative student-level precision** meets the floor.

### Is taking the final temporal prediction fair?
Yes, **if “final” means the latest landmark available at the current decision cutoff**, and the same rule is used in validation and test. It is not fair to use the last row of the student's complete observed history retrospectively when that row occurs after the decision date. It is also not fair to tune thresholds using any-landmark aggregation and then deploy latest-landmark scoring; those are different policies.